In [1]:
# TP 5 — Projet de synthèse : carnet de notes
import json
from datetime import datetime
from functools import wraps

etudiants = {
    "L3-001": {
        "nom": "Rakoto",
        "prenom": "Soa",
        "notes": {"Python": [15, 14], "Réseaux": [12]},
    },
    "L3-002": {
        "nom": "Rabe",
        "prenom": "Andry",
        "notes": {"Python": [9, 13], "Réseaux": [11, 10]},
    },
}
historique = []


# Décorateur : journaliser les appels des fonctions qui modifient les données
def journaliser(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        historique.append((
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            fonction.__name__,
            {"args": repr(args), "kwargs": repr(kwargs)}
        ))
        return fonction(*args, **kwargs)
    return enveloppe


# Décorateur : vérifier que la note est entre 0 et 20
def note_valide(fonction):
    @wraps(fonction)
    def enveloppe(matricule, matiere, note, *args, **kwargs):
        if not isinstance(note, (int, float)) or not 0 <= note <= 20:
            raise ValueError("La note doit être un nombre compris entre 0 et 20.")
        return fonction(matricule, matiere, note, *args, **kwargs)
    return enveloppe


# Décorateur : vérifier que le premier argument matricule existe
def etudiant_existe(fonction):
    @wraps(fonction)
    def enveloppe(matricule, *args, **kwargs):
        if matricule not in etudiants:
            raise KeyError(f"Aucun étudiant ne possède le matricule {matricule}.")
        return fonction(matricule, *args, **kwargs)
    return enveloppe


@journaliser
def ajouter_etudiant(matricule, nom, prenom):
    if matricule in etudiants:
        return False, f"Le matricule {matricule} existe déjà."
    etudiants[matricule] = {"nom": nom, "prenom": prenom, "notes": {}}
    return True, f"Étudiant {prenom} {nom} ajouté."


@journaliser
@etudiant_existe
@note_valide
def ajouter_note(matricule, matiere, note):
    notes_matiere = etudiants[matricule]["notes"].setdefault(matiere, [])
    notes_matiere.append(float(note))
    return True, f"Note {note}/20 ajoutée en {matiere}."


@etudiant_existe
def moyenne_matiere(matricule, matiere):
    notes = etudiants[matricule]["notes"].get(matiere, [])
    return sum(notes) / len(notes) if notes else None


@etudiant_existe
def moyenne_generale(matricule):
    moyennes = [
        sum(notes) / len(notes)
        for notes in etudiants[matricule]["notes"].values()
        if notes
    ]
    return sum(moyennes) / len(moyennes) if moyennes else 0.0


def classement():
    resultat = []
    for matricule, etudiant in etudiants.items():
        moyenne = moyenne_generale(matricule)
        nom_complet = f"{etudiant['prenom']} {etudiant['nom']}"
        resultat.append((nom_complet, moyenne))
    return sorted(resultat, key=lambda element: element[1], reverse=True)


def majors():
    matieres = {
        matiere
        for etudiant in etudiants.values()
        for matiere in etudiant["notes"]
    }
    resultat = {}
    for matiere in sorted(matieres):
        candidats = []
        for etudiant in etudiants.values():
            notes = etudiant["notes"].get(matiere, [])
            if notes:
                moyenne = sum(notes) / len(notes)
                nom_complet = f"{etudiant['prenom']} {etudiant['nom']}"
                candidats.append((nom_complet, moyenne))
        if candidats:
            resultat[matiere] = max(candidats, key=lambda element: element[1])[0]
    return resultat


def afficher_bulletin(matricule):
    if matricule not in etudiants:
        print("Matricule introuvable.")
        return
    etudiant = etudiants[matricule]
    print(f"\nBulletin de {etudiant['prenom']} {etudiant['nom']} ({matricule})")
    if not etudiant["notes"]:
        print("Aucune note enregistrée.")
    for matiere, notes in etudiant["notes"].items():
        moyenne = sum(notes) / len(notes) if notes else 0
        print(f"- {matiere} : notes={notes}, moyenne={moyenne:.2f}")
    print(f"Moyenne générale : {moyenne_generale(matricule):.2f}/20")


def sauvegarder(fichier="carnet_notes.json"):
    with open(fichier, "w", encoding="utf-8") as f:
        json.dump(
            {"etudiants": etudiants, "historique": historique},
            f, ensure_ascii=False, indent=2
        )
    print(f"Carnet sauvegardé dans {fichier}.")


def charger(fichier="carnet_notes.json"):
    global etudiants, historique
    with open(fichier, "r", encoding="utf-8") as f:
        donnees = json.load(f)
    etudiants = donnees["etudiants"]
    historique = donnees.get("historique", [])
    print(f"Carnet chargé depuis {fichier}.")


def afficher_classement():
    print("\nClassement général")
    for position, (nom, moyenne) in enumerate(classement(), start=1):
        print(f"{position}. {nom:<22} {moyenne:.2f}")


def afficher_majors():
    print("\nMeilleurs étudiants par matière")
    resultats = majors()
    if not resultats:
        print("Aucune note disponible.")
    for matiere, nom in resultats.items():
        print(f"- {matiere} : {nom}")


def afficher_historique():
    print("\nHistorique des modifications")
    if not historique:
        print("Aucune opération enregistrée.")
    for date_heure, nom_fonction, arguments in historique:
        print(f"{date_heure} | {nom_fonction} | {arguments}")


def demander_nombre(message):
    while True:
        try:
            return float(input(message))
        except ValueError:
            print("Veuillez saisir un nombre valide.")


def menu():
    while True:
        print("""
========== CARNET DE NOTES ==========
1. Ajouter un étudiant
2. Ajouter une note
3. Afficher un bulletin
4. Afficher le classement
5. Afficher les majors
6. Afficher l'historique
7. Sauvegarder
8. Charger
0. Quitter
""")
        choix = input("Votre choix : ").strip()

        try:
            if choix == "1":
                matricule = input("Matricule : ").strip()
                nom = input("Nom : ").strip()
                prenom = input("Prénom : ").strip()
                succes, message = ajouter_etudiant(matricule, nom, prenom)
                print(message)

            elif choix == "2":
                matricule = input("Matricule : ").strip()
                matiere = input("Matière : ").strip()
                note = demander_nombre("Note sur 20 : ")
                succes, message = ajouter_note(matricule, matiere, note)
                print(message)

            elif choix == "3":
                afficher_bulletin(input("Matricule : ").strip())

            elif choix == "4":
                afficher_classement()

            elif choix == "5":
                afficher_majors()

            elif choix == "6":
                afficher_historique()

            elif choix == "7":
                fichier = input("Nom du fichier [carnet_notes.json] : ").strip()
                sauvegarder(fichier or "carnet_notes.json")

            elif choix == "8":
                fichier = input("Nom du fichier [carnet_notes.json] : ").strip()
                charger(fichier or "carnet_notes.json")

            elif choix == "0":
                print("Fin du programme.")
                break

            else:
                print("Choix invalide.")

        except (KeyError, ValueError, OSError, json.JSONDecodeError) as erreur:
            print("Erreur :", erreur)


if __name__ == "__main__":
    menu()



========== CARNET DE NOTES ==========
1. Ajouter un étudiant
2. Ajouter une note
3. Afficher un bulletin
4. Afficher le classement
5. Afficher les majors
6. Afficher l'historique
7. Sauvegarder
8. Charger
0. Quitter



Votre choix :  1
Matricule :  4587
Nom :  andry
Prénom :  fanantena


Étudiant fanantena andry ajouté.

========== CARNET DE NOTES ==========
1. Ajouter un étudiant
2. Ajouter une note
3. Afficher un bulletin
4. Afficher le classement
5. Afficher les majors
6. Afficher l'historique
7. Sauvegarder
8. Charger
0. Quitter



Votre choix :  6



Historique des modifications
2026-10-13 19:12:27 | ajouter_etudiant | {'args': "('4587', 'andry', 'fanantena')", 'kwargs': '{}'}

========== CARNET DE NOTES ==========
1. Ajouter un étudiant
2. Ajouter une note
3. Afficher un bulletin
4. Afficher le classement
5. Afficher les majors
6. Afficher l'historique
7. Sauvegarder
8. Charger
0. Quitter



...

Note 16.0/20 ajoutée en anglais.

========== CARNET DE NOTES ==========
1. Ajouter un étudiant
2. Ajouter une note
3. Afficher un bulletin
4. Afficher le classement
5. Afficher les majors
6. Afficher l'historique
7. Sauvegarder
8. Charger
0. Quitter



Votre choix :  6



Historique des modifications
2026-10-13 19:12:27 | ajouter_etudiant | {'args': "('4587', 'andry', 'fanantena')", 'kwargs': '{}'}
2026-10-13 19:13:10 | ajouter_note | {'args': "('4587', 'anglais', 16.0)", 'kwargs': '{}'}

========== CARNET DE NOTES ==========
1. Ajouter un étudiant
2. Ajouter une note
3. Afficher un bulletin
4. Afficher le classement
5. Afficher les majors
6. Afficher l'historique
7. Sauvegarder
8. Charger
0. Quitter



Votre choix :  0


Fin du programme.
